# Devoir 1 : Classification de sentiments de deux façons

Bienvenue à votre premier devoir ! Lorsque vous le soumettez sur ZoneCours, assurez-vous de renommer le fichier avec votre nom comme suit : **"Assignment1_{firstname}_{lastname}.ipynb"**. Soumettez une version du notebook où vous avez fait roulez toutes les cellules et laissez les sorties apparentes. La date de remise est fixée à 23 h 59 le 26 février. Le devoir est noté sur un total de 42 points, avec 2 points boni possibles. Si vous avez travaillé sur une quelconque partie du devoir avec d’autres personnes, veuillez indiquer leurs noms ici :

In [1]:
__author__ = "{Votre nom}"
__collaborators__ = "{leurs noms, si plus qu'un, séparez les par un ; si non, laissez vide}"

Toutes les bibliothèques dont vous aurez besoin sont listées ici. Si certaines bibliothèques qui pourraient sembler pertinentes ont été exclues, c’est intentionnel afin que vous preniez le temps d’écrire vos propres fonctions. Veuillez ne pas modifier cette cellule et abstenez-vous d’ajouter d’autres bibliothèques ailleurs. Vous pouvez utiliser toute fonction que vous jugez utile provenant de ces bibliothèques.

In [1]:
from nltk.corpus import movie_reviews
import string
import re
import random
import math
from collections import Counter
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
random.seed(202601)
torch.manual_seed(202401)
device = torch.device('cpu')

Nous allons comparer deux types de classificateurs de critiques de films : (1) un classificateur de régression logistique basé sur des N-grammes et (2) un classificateur basé sur un LSTM. La première étape consiste à télécharger les données et à créer nos ensembles d’entraînement et de test. Il y a deux classes de critiques : positives et négatives.

In [2]:
review_classes = movie_reviews.categories()
pos_reviews = [(movie_reviews.raw(fileid), 'pos') for fileid in movie_reviews.fileids(categories=['pos'])]
neg_reviews = [(movie_reviews.raw(fileid), 'neg') for fileid in movie_reviews.fileids(categories=['neg'])]
train_dataset = pos_reviews[:800] + neg_reviews[:800]
test_dataset = pos_reviews[800:] + neg_reviews[800:]

In [3]:
train_dataset[0]

('films adapted from comic books have had plenty of success , whether they\'re about superheroes ( batman , superman , spawn ) , or geared toward kids ( casper ) or the arthouse crowd ( ghost world ) , but there\'s never really been a comic book like from hell before . \nfor starters , it was created by alan moore ( and eddie campbell ) , who brought the medium to a whole new level in the mid \'80s with a 12-part series called the watchmen . \nto say moore and campbell thoroughly researched the subject of jack the ripper would be like saying michael jackson is starting to look a little odd . \nthe book ( or " graphic novel , " if you will ) is over 500 pages long and includes nearly 30 more that consist of nothing but footnotes . \nin other words , don\'t dismiss this film because of its source . \nif you can get past the whole comic book thing , you might find another stumbling block in from hell\'s directors , albert and allen hughes . \ngetting the hughes brothers to direct this see

In [4]:
len(train_dataset)

1600

1. **[3 points]** L’étape suivante consiste à tokeniser les critiques. Vous devez maintenant écrire une fonction de prétraitement qui prend une critique et (1) supprime la ponctuation, (2) met tous les mots en minuscules et (3) les sépare en tokens en utilisant tout type d’espacement. Assurez-vous qu’il n’y a pas de tokens vides.

In [5]:
def preprocess(review:str):
    tokenized_review = []
    punctuations = list(string.punctuation)
    ## TO DO
    ### Suppression ponctuation
    punctuationsStr = "".join(punctuations)
    pattern = f"[{re.escape(punctuationsStr)}]+"
    reviewPrepared = re.sub(pattern,'',review)
    ### Cast en minuscules
    reviewPrepared= reviewPrepared.lower()
    ### Séparation en token
    tokenized_review = reviewPrepared.split()

    if "" in tokenized_review:
        raise ValueError("ATTENTION: tokenized_review contient des tokens vides") 

    
    ##
    return tokenized_review

In [6]:
# TODO: à supprimer avant de soumettre
# Pour tester la fonction preprocess:
## échantillon de données d'entraînement déterministique
#preprocess(train_dataset[0][0])

## échantillon de données d'entraînement aléatoire
rndTrainDataIdx = random.randint(0, len(train_dataset)-1)
rndPreprocessData=preprocess(train_dataset[rndTrainDataIdx][0])
rndPreprocessData

['youve',
 'got',
 'mail',
 'works',
 'alot',
 'better',
 'than',
 'it',
 'deserves',
 'to',
 'in',
 'order',
 'to',
 'make',
 'the',
 'film',
 'a',
 'success',
 'all',
 'they',
 'had',
 'to',
 'do',
 'was',
 'cast',
 'two',
 'extremely',
 'popular',
 'and',
 'attractive',
 'stars',
 'have',
 'them',
 'share',
 'the',
 'screen',
 'for',
 'about',
 'two',
 'hours',
 'and',
 'then',
 'collect',
 'the',
 'profits',
 'no',
 'real',
 'acting',
 'was',
 'involved',
 'and',
 'there',
 'is',
 'not',
 'an',
 'original',
 'or',
 'inventive',
 'bone',
 'in',
 'its',
 'body',
 'its',
 'basically',
 'a',
 'complete',
 'reshoot',
 'of',
 'the',
 'shop',
 'around',
 'the',
 'corner',
 'only',
 'adding',
 'a',
 'few',
 'modern',
 'twists',
 'essentially',
 'it',
 'goes',
 'against',
 'and',
 'defies',
 'all',
 'concepts',
 'of',
 'good',
 'contemporary',
 'filmmaking',
 'its',
 'overly',
 'sentimental',
 'and',
 'at',
 'times',
 'terribly',
 'mushy',
 'not',
 'to',
 'mention',
 'very',
 'manipulative'

## Classificateur 1 : Caractéristiques N‑grammes et régression logistique

Ce premier classificateur extrait des caractéristiques de type N‑grammes à partir des critiques, les transforme en vecteurs, puis entraîne un classificateur basé sur une régression logistique à partir de ces caractéristiques.

2. **[3 points]** Écrivez une fonction utilitaire qui retourne un vocabulaire de N‑grammes, *ngram_vocab*, pour toute valeur de *n* ≥ 1, étant donné un jeu de données *tokenized_reviews* étiquetées (une liste de tuples *(tokenized_review, review_class)*). Il est important que la taille du vocabulaire soit limitée à la taille maximale *max_vocab_size*. *ngram_vocab* doit inclure les *max_vocab_size* N‑grams les plus fréquents sur l’ensemble des critiques. Le type de retour doit être une liste de tuples, où chaque tuple représente un N‑gramme.

In [7]:
def get_ngram_tokenized(tokenized_review:list, n:int) -> list[tuple]:
    """
    Fonction qui construit un token de N-grammes à partir d'un token de mots.
    Args:
        tokenized_review : list 
            Token de mots d'une critique.
        n : int
            La taille des N-grammes à construire.
    """
    grams = [tokenized_review[i:] for i in range(n)]
    return list(zip(*grams))

In [8]:
# TODO: à supprimer avant de soumettre
# Pour tester la fonction get_ngram_tokenized
get_ngram_tokenized(['this', 'is', 'a', 'test'], 3)
get_ngram_tokenized(rndPreprocessData, 10)

[('youve',
  'got',
  'mail',
  'works',
  'alot',
  'better',
  'than',
  'it',
  'deserves',
  'to'),
 ('got',
  'mail',
  'works',
  'alot',
  'better',
  'than',
  'it',
  'deserves',
  'to',
  'in'),
 ('mail',
  'works',
  'alot',
  'better',
  'than',
  'it',
  'deserves',
  'to',
  'in',
  'order'),
 ('works',
  'alot',
  'better',
  'than',
  'it',
  'deserves',
  'to',
  'in',
  'order',
  'to'),
 ('alot',
  'better',
  'than',
  'it',
  'deserves',
  'to',
  'in',
  'order',
  'to',
  'make'),
 ('better',
  'than',
  'it',
  'deserves',
  'to',
  'in',
  'order',
  'to',
  'make',
  'the'),
 ('than', 'it', 'deserves', 'to', 'in', 'order', 'to', 'make', 'the', 'film'),
 ('it', 'deserves', 'to', 'in', 'order', 'to', 'make', 'the', 'film', 'a'),
 ('deserves',
  'to',
  'in',
  'order',
  'to',
  'make',
  'the',
  'film',
  'a',
  'success'),
 ('to', 'in', 'order', 'to', 'make', 'the', 'film', 'a', 'success', 'all'),
 ('in', 'order', 'to', 'make', 'the', 'film', 'a', 'success', 

In [9]:
def get_ngram_vocabulary_maxsize(tokenized_reviews:list[tuple[list[str], str]], n:int, max_vocab_size:int):
    ngram_vocab = []
    ## TO DO
    ### 1) Construire token N-grammes par critique
    ngrams_tokenized_reviews = []
    for review, _ in tokenized_reviews:
        ngrams_tokenized_reviews.extend(get_ngram_tokenized(review, n))
    ### 2) Faire le compte des N-grammes sur l'ensemble des critiques
    ngramsTokenizedReviewsCounter = Counter(ngrams_tokenized_reviews)
    ### 3) Le Top-N définit le nouveau vocabulaire.
    ngramsTokenizedReviewsCounter = ngramsTokenizedReviewsCounter.most_common(max_vocab_size)
    ngram_vocab = list(list(zip(*ngramsTokenizedReviewsCounter))[0])

    
    ##
    return ngram_vocab

In [10]:
# TODO: à supprimer avant de soumettre
# Pour tester la fonction get_ngram_vocabulary_maxsize
tokenized_reviews = [(preprocess(review), label) for review, label in train_dataset]
get_ngram_vocabulary_maxsize(tokenized_reviews, 3, 50)


[('one', 'of', 'the'),
 ('of', 'the', 'film'),
 ('in', 'the', 'film'),
 ('the', 'film', 'is'),
 ('a', 'lot', 'of'),
 ('some', 'of', 'the'),
 ('the', 'rest', 'of'),
 ('to', 'be', 'a'),
 ('of', 'the', 'movie'),
 ('the', 'fact', 'that'),
 ('in', 'this', 'film'),
 ('this', 'is', 'a'),
 ('rest', 'of', 'the'),
 ('most', 'of', 'the'),
 ('this', 'film', 'is'),
 ('is', 'one', 'of'),
 ('the', 'movie', 'is'),
 ('of', 'the', 'most'),
 ('in', 'the', 'movie'),
 ('the', 'end', 'of'),
 ('out', 'of', 'the'),
 ('in', 'order', 'to'),
 ('seems', 'to', 'be'),
 ('as', 'well', 'as'),
 ('this', 'is', 'the'),
 ('could', 'have', 'been'),
 ('it', 'is', 'a'),
 ('there', 'is', 'a'),
 ('a', 'couple', 'of'),
 ('of', 'the', 'best'),
 ('in', 'this', 'movie'),
 ('to', 'be', 'the'),
 ('part', 'of', 'the'),
 ('supposed', 'to', 'be'),
 ('end', 'of', 'the'),
 ('all', 'of', 'the'),
 ('at', 'the', 'end'),
 ('a', 'group', 'of'),
 ('one', 'of', 'those'),
 ('the', 'story', 'is'),
 ('there', 'is', 'no'),
 ('in', 'the', 'end'),
 

3. **[4 points]** Écrivez une fonction utilitaire qui prend un vocabulaire de N‑grammes, *ngram_vocab*, ainsi qu’une critique tokenisée, *tokenized_review*, et qui retourne une représentation vectorisée des comptes de N‑grammes dans la critique. Le type de retour doit être un torch tensor de float, où chaque indice correspond à un N‑gramme dans *ngram_vocab* et où les valeurs correspondent à leurs comptes respectifs.

In [11]:
def get_vectorized_review(tokenized_review:list[str], ngram_vocab:list[tuple]):
    vectorized_review = torch.Tensor([0.0])
    ## TO DO
    ngram_size = len(ngram_vocab[0])
    ### Construire token N-grammes de la critique
    review_ngrams = get_ngram_tokenized(tokenized_review, ngram_size)
    ### Compter les N-grammes de la critique 
    review_ngramsCounter = Counter(review_ngrams)

    vectorized_review = torch.zeros(len(ngram_vocab), dtype=torch.float)
    for i, ngram in enumerate(ngram_vocab):
        if ngram in review_ngramsCounter:
            vectorized_review[i] = review_ngramsCounter[ngram]

    
    ##
    return vectorized_review

In [12]:
# TODO: à supprimer avant de soumettre
# Pour tester la fonction get_vectorized_review
get_vectorized_review(rndPreprocessData, get_ngram_vocabulary_maxsize(tokenized_reviews, 2, 20))

tensor([3., 1., 1., 0., 2., 0., 0., 1., 0., 0., 3., 1., 2., 0., 1., 0., 0., 1.,
        0., 0.])

4. **[4 points]** Écrivez une classe *NgramReviewDataset* qui implémente un objet *torch Dataset*. Elle doit prendre comme input *tokenized_reviews* (une liste de tuples *(tokenized_review, review_class)*) ainsi qu’un vocabulaire de N‑grammes, *ngram_vocab*. Votre fonction d’initialisation doit vectoriser toutes les critiques, de sorte que *self.vectorized_reviews* soit un torch tensor (ou une matrice) de taille |tokenized_reviews| × |ngram_vocab|. Quant à *self.labels*, il doit s’agir d’un torch tensor de taille |tokenized_reviews| × |review_classes|, où chaque élément est un vecteur one‑hot tel que [1,0] correspond à la classe « neg » et [0,1] à la classe « pos ».

In [13]:
class NgramReviewDataset(Dataset):
    def __init__(self, tokenized_reviews:list[tuple[list[str], str]], ngram_vocab:list[tuple]):
        vectorized_reviews = [] 
        labels = [] 
        ## TO DO
        for review, label in tokenized_reviews:
            vectorized_reviews.append(get_vectorized_review(review, ngram_vocab))
            labels.append(0 if label == 'neg' else 1)
        
        ##
        self.vectorized_reviews = torch.stack(vectorized_reviews, dim=0)
        self.labels = F.one_hot(torch.Tensor(labels).long()).float()
        self.length = len(labels)

    def __getitem__(self, index):
        ## TO DO
        vectorized_review = self.vectorized_reviews[index]
        label = self.labels[index]       
        
        ##
        return vectorized_review, label, index

    def __len__(self):
        return self.length

5. **[2 points]** Écrivez une fonction *get_ngram_dataloader* qui prend en entrée *tokenized_reviews*, *ngram_vocab*, une taille de lot *batch_size* et un hyperparamètre *shuffle*. Elle doit utiliser la classe *NgramReviewDataset* définie précédemment et retourner un objet de type *torch.utils.data.DataLoader* qui tient compte des paramètres d’entrée de la fonction.

In [14]:
def get_ngram_dataloader(tokenized_reviews:list[tuple[list[str], str]], ngram_vocab:list[tuple], batch_size:int, shuffle:bool):
    ## TO DO
    data = NgramReviewDataset(tokenized_reviews, ngram_vocab)
    dataloader = DataLoader(data, batch_size=batch_size, shuffle=shuffle)

    
    ##
    return dataloader

#### Régression logistique avec un réseau neuronal à une seule couche linéaire

Voici notre classe de modèle de régression logistique (LR) que nous utiliserons comme base pour nos classificateurs N‑grammes LR.

In [15]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        
    def forward(self, x):
        return self.linear(x)

6. **[4 points]** Complétez la fonction *train* que nous utiliserons à la fois pour notre classificateur N‑grammes LR et pour le classificateur basé sur un LSTM. Dans les deux cas, nous effectuons une régression logistique sur un ensemble de caractéristiques variables ; la fonction de perte utilisée dans votre fonction *train* doit donc être choisie en conséquence.

In [16]:
def train(model, dataloader, lr = 0.001, epochs = 5):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        train_loop = tqdm(dataloader, desc=f"Epoch {epoch+1}")
        for i, (batch_inputs, batch_labels, _) in enumerate(train_loop):
            train_loop.set_description(f"Epoch {epoch+1} | Batch {i}")
            ## TO DO
            ### Clear gradients from optimizer
            optimizer.zero_grad()

            ### Forward pass
            outputs = model(batch_inputs)

            ### Compute loss
            loss = nn.CrossEntropyLoss()(outputs, batch_labels.argmax(dim=1))

            ### Backward pass
            loss.backward()

            ### Update parameters
            optimizer.step()

            
            ##
    return model  

Entraînons maintenant trois classificateurs N‑grammes en utilisant respectivement des caractéristiques unigrammes, bigrammes et trigrammes.

In [17]:
def train_LR_ngram_classifier(train_dataset:list[tuple[str, str]], review_classes:list[str], n:int, batch_size = 32): 
    tokenized_train = [(preprocess(review), review_class) for review, review_class in  train_dataset]
    ngram_vocab = get_ngram_vocabulary_maxsize(tokenized_train, n, 30000)
    
    train_dataloader = get_ngram_dataloader(tokenized_train, ngram_vocab, batch_size, shuffle=True)

    model = LogisticRegression(len(ngram_vocab), len(review_classes))
    trained_model = train(model, train_dataloader)
    return ngram_vocab, trained_model

In [18]:
unigram_vocab, unigram_model = train_LR_ngram_classifier(train_dataset, review_classes, 1)
bigram_vocab, bigram_model = train_LR_ngram_classifier(train_dataset, review_classes, 2)
trigram_vocab, trigram_model = train_LR_ngram_classifier(train_dataset, review_classes, 3)

Epoch 1:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/50 [00:00<?, ?it/s]

...

Epoch 1:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/50 [00:00<?, ?it/s]

#### Évaluation de nos classificateurs LR
Voici la fonction d’évaluation pour nos classificateurs. Elle prend le *test_dataset* ainsi qu’un modèle de classification entraîné et retourne les prédictions de classe du modèle pour chaque critique dans les données de test. Elle peut être utilisée pour les deux types de classificateurs présentés dans ce devoir.

In [19]:
def eval(model, dataloader):
    ids = []
    labels = []
    predictions = []
    model.eval()
    with torch.no_grad():
        for batch_inputs, batch_labels, batch_ids in dataloader:
            x = batch_inputs.to(device)
            y_pred = model(x)
            labels += batch_labels.tolist()
            predictions += y_pred.tolist()
            if not isinstance(batch_ids, tuple):
                batch_ids.tolist()
            ids += batch_ids
            
    results = [{'review_id':idx, 'class_label':label, 'prediction': pred} for idx, label, pred in zip(ids, labels, predictions)]
    return results

Évaluons maintenant nos classificateurs N‑grammes sur les données de test.

In [20]:
def eval_LR_ngram_classifier(test_dataset:list[tuple[str, str]], ngram_vocab:list[tuple], model):
    tokenized_test = [(preprocess(review), review_class) for review, review_class in  test_dataset]
    test_dataloader = get_ngram_dataloader(tokenized_test, ngram_vocab, batch_size=32, shuffle=False)
    results = eval(model, test_dataloader)
    return results

In [21]:
unigram_results = eval_LR_ngram_classifier(test_dataset, unigram_vocab, unigram_model)
bigram_results = eval_LR_ngram_classifier(test_dataset, bigram_vocab, bigram_model)
trigram_results = eval_LR_ngram_classifier(test_dataset, trigram_vocab, trigram_model)

#### Résultats

Enfin, comparons les performances relatives de nos classificateurs LR unigramme, bigramme et trigramme.

7. **[4 points]** Écrivez une fonction qui retourne un dictionnaire contenant la précision, le rappel et le score F1 globaux pour toutes les classes, ainsi que l’exactitude (accuracy) globale d’un classificateur.

In [22]:
def get_accuracy_scores(results:list[dict]):
    accuracy = dict([('precision', 0.0),('recall', 0.0), ('f1', 0.0), ('accuracy', 0.0)])
    classes = {'pos':1, 'neg':0}
    ## TO DO
    TP = FP = TN = FN = 0
    for result in results:
        class_label = result['class_label']
        prediction = result['prediction']
        true_label = class_label.index(max(class_label))
        pred_label = prediction.index(max(prediction))
        if true_label == 1 and pred_label == 1:
            TP += 1
        elif true_label == 0 and pred_label == 1:
            FP += 1
        elif true_label == 0 and pred_label == 0:
            TN += 1
        elif true_label == 1 and pred_label == 0:
            FN += 1

    
    accuracy['accuracy'] = (TP + TN) / (TP + FP + TN + FN)
    accuracy['precision'] = TP / (TP + FP) 
    accuracy['recall'] = TP / (TP + FN) 
    accuracy['f1'] = 2 * TP / (2 * TP + FP + FN) 
    ##
    return accuracy

Comparons maintenant les resultats de nos trois classificateurs N-grammes.

In [23]:
accuracies = {'unigram':get_accuracy_scores(unigram_results),
                  'bigram':get_accuracy_scores(bigram_results),
                  'trigram':get_accuracy_scores(trigram_results)}

In [24]:
pd.DataFrame.from_dict(accuracies, orient='index')

,precision,recall,f1,accuracy
unigram,0.890547,0.895,0.892768,0.8925
bigram,0.895833,0.860,0.877551,0.8800
trigram,0.839080,0.730,0.780749,0.7950


## Classificateur 2 : Classificateur basé sur un LSTM

Nous allons entraîner un classificateur basé sur un LSTM en utilisant l’état final de sortie du LSTM comme caractéristiques pour la classification des critiques.

8. **[3 points]** Écrivez une fonction utilitaire qui retourne le vocabulaire de tokens, *vocab*, étant donné un jeu de données de *tokenized_reviews* étiquetées (une liste de tuples *(tokenized_review, review_class)*). Il est important que la taille du vocabulaire soit limitée à *max_vocab_size*. *vocab* doit inclure les *max_vocab_size* tokens uniques les plus fréquents sur l’ensemble des critiques. Le type de retour sera différent du problème 2 : cette fois, *vocab* doit être un dictionnaire où les clés sont les types de tokens et les valeurs sont leurs indices, ordonnés par fréquence. Notez que les deux premiers éléments du dictionnaire *vocab* sont les tokens spéciaux '\[PAD\]' pour le remplissage (padding) du texte et '\[UNK\]' pour les mots inconnus, qui ont respectivement les indices réservés 0 et 1. Ainsi, l’indice du token le plus fréquent doit commencer à 2.

In [25]:
def get_vocabulary_maxsize(tokenized_reviews:list[tuple[list[str], str]], max_vocab_size:int):
    vocab = dict([('[PAD]', 0), ('[UNK]',1)])
    ## TO DO
    ### Construire token de l'ensemble des mots de toutes les critiques
    tokensOfEachReview=list(zip(*tokenized_reviews))[0]
    tokens = [token for review in tokensOfEachReview for token in review]
    ### Faire le compte des mots sur l'ensemble des critiques
    tokensCounter = Counter(tokens)
    ### Le Top-N définit le nouveau vocabulaire.
    tokensCounter = tokensCounter.most_common(max_vocab_size)
    vocab.update({token: idx+2 for idx, (token, _) in enumerate(tokensCounter)})

    
    ##
    return vocab

In [26]:
# TODO: à supprimer avant de soumettre
# Pour tester la fonction get_vocabulary_maxsize
get_vocabulary_maxsize(tokenized_reviews, 10000)

{'[PAD]': 0,
 '[UNK]': 1,
 'the': 2,
 'a': 3,
 'and': 4,
 'of': 5,
 'to': 6,
 'is': 7,
 'in': 8,
 'that': 9,
 'it': 10,
 'as': 11,
 'with': 12,
 'for': 13,
 'his': 14,
 'this': 15,
 'film': 16,
 'but': 17,
 'he': 18,
 'i': 19,
 'on': 20,
 'are': 21,
 'by': 22,
 'be': 23,
 'its': 24,
 'an': 25,
 'not': 26,
 'one': 27,
 'who': 28,
 'movie': 29,
 'at': 30,
 'was': 31,
 'from': 32,
 'have': 33,
 'has': 34,
 'you': 35,
 'her': 36,
 'they': 37,
 'all': 38,
 'so': 39,
 'like': 40,
 'about': 41,
 'out': 42,
 'more': 43,
 'when': 44,
 'which': 45,
 'up': 46,
 'their': 47,
 'what': 48,
 'or': 49,
 'some': 50,
 'just': 51,
 'if': 52,
 'there': 53,
 'into': 54,
 'him': 55,
 'she': 56,
 'even': 57,
 'only': 58,
 'than': 59,
 'no': 60,
 'good': 61,
 'we': 62,
 'most': 63,
 'time': 64,
 'can': 65,
 'will': 66,
 'story': 67,
 'films': 68,
 'been': 69,
 'would': 70,
 'much': 71,
 'also': 72,
 'get': 73,
 'other': 74,
 'characters': 75,
 'do': 76,
 'very': 77,
 'character': 78,
 'them': 79,
 'two': 80,


9. **[3 points]** Écrivez une fonction utilitaire qui prend un dictionnaire de vocabulaire, *vocab*, et une critique tokenisée, *tokenized_review*, et qui retourne une *indexed_review*, où chaque token a été remplacé par son indice correspondant dans le dictionnaire *vocab*. Si un mot n’est pas présent dans le vocabulaire, il doit être remplacé par l’indice du token spécial '\[UNK\]'. Le type de retour doit être un torch tensor de int64.

In [27]:
def get_indexed_review(tokenized_review:list[str], vocab:dict[str,int]):
    indexed_review = torch.tensor([0], dtype=torch.long)
    ## TO DO
    tokenized_review_idx = [vocab.get(token, vocab['[UNK]']) for token in tokenized_review]
    indexed_review = torch.cat((indexed_review, torch.tensor(tokenized_review_idx, dtype=torch.long)), dim=0)
    
    
    ##
    return indexed_review

In [28]:
#TODO: à supprimer avant de soumettre
# Pour tester la fonction get_indexed_review
vocabTest = get_vocabulary_maxsize(tokenized_reviews, 50)
indexed_reviewTest = get_indexed_review(tokenized_reviews[random.randint(0, len(tokenized_reviews) - 1)][0], vocabTest)
indexed_reviewTest

tensor([ 0,  2,  1,  1,  1, 22,  1,  1,  7,  1, 27,  5,  2,  1,  1,  1,  1,  3,
         1,  1, 41,  2,  1,  5,  3,  1,  5,  1, 32,  3,  1,  1,  1,  1,  2,  1,
         1,  1,  6, 23, 27,  5,  1,  1,  9, 35,  1,  1,  1,  8, 15,  1,  1,  1,
        13,  1,  1,  5,  1,  1,  1, 21,  1,  8, 22, 38,  1,  4,  1,  1,  1,  1,
         2,  1,  1,  1,  2,  1,  1,  2,  1,  1,  5,  2, 16,  1,  1, 12, 25,  1,
         1,  1,  2,  1,  5,  3,  1,  5,  1, 32,  3,  1,  1, 28,  1,  4,  1, 42,
         1,  1,  1,  1, 22,  1,  1,  1,  7,  1,  1, 20, 11,  2,  1,  1,  5,  2,
         1,  1,  3,  1,  1,  2,  1,  1, 47,  1, 30,  3,  1,  6,  1,  1,  6,  2,
         1,  1,  1,  1,  1,  1,  1,  1,  5,  2,  1,  1,  1,  1,  1,  4,  1,  1,
         1,  1, 23,  1,  1, 13,  1, 17, 26,  1,  1,  1,  1,  1,  2,  1,  2,  1,
         1,  1,  1,  1,  1, 27,  1,  1, 20,  3,  1,  1, 27,  5, 47,  1,  1, 15,
         1,  5,  1,  1,  1,  6,  1,  2,  1,  5,  3,  1,  1,  5,  1, 28,  1, 23,
         1,  4,  1,  1, 22,  2,  1,  4, 

10. **[4 points]** Écrivez une classe *LSTMReviewDataset* qui implémente un objet *torch Dataset*. Elle doit prendre en entrée *tokenized_reviews* (une liste de tuples *(tokenized_review, review_class)*) ainsi qu’un dictionnaire de vocabulaire, *vocab*. Votre fonction d’initialisation doit transformer toutes les critiques tokenisées en séquences d’indices, de sorte que *self.indexed_reviews* soit une liste de torch tensor. Quant à *self.labels*, il doit également s’agir d’un torch tensor de taille |tokenized_reviews| × |review_classes|, où chaque élément est un vecteur one‑hot tel que [1,0] correspond à la classe « neg » et [0,1] à la classe « pos ».

In [34]:
class LSTMReviewDataset(Dataset):
    def __init__(self, tokenized_reviews:list[tuple[list[str], str]], vocab:dict[str,int]):
        indexed_reviews = [] 
        labels = [] 
        ## TO DO
        for tokenized_review, label in tokenized_reviews:
            indexed_reviews.append(get_indexed_review(tokenized_review, vocab))
            labels.append(0 if label == 'neg' else 1)
        
              
        ##
        self.indexed_reviews = indexed_reviews
        self.labels = F.one_hot(torch.Tensor(labels).long()).float()
        self.length = len(labels)

    def __getitem__(self, idx):
        ## TO DO
        indexed_review = self.indexed_reviews[idx]
        label = self.labels[idx]

        
        ##
        return indexed_review, label, idx

    def __len__(self):
        return self.length

11. **[3 points]** Écrivez une fonction *collate* personnalisée. Cette fonction sert à standardiser tous les éléments d’un lot. Elle peut être passée au *dataloader* comme paramètre. Comme chaque élément est une séquence d’indices représentant une critique, les séquences peuvent avoir des longueurs différentes. Afin de les regrouper dans un seul tenseur, nous devons standardiser leurs longueurs pour que toutes les séquences d’un lot aient la même longueur, égale à celle de la séquence la plus longue du lot. Nous ferons cela en remplissant les séquences avec des zéros — l’indice du token spécial '\[PAD\]' — au début de chaque séquence. Par exemple, si la longueur maximale d’un lot est 5 et que nous avons la séquence [22,300,6584], elle deviendra [0,0,22,300,6584]. Les sorties finales *batched_sequences* et *batched_labels* doivent être des torch tensor de type *long*, avec les dimensions |batch_size| × |taille maximale de séquence| et |batch_size| × |output_dim|, où *output_dim* correspond au nombre de classes. *batched_ids* vous est déjà fourni.

In [40]:
def collate_fn(items):
    sequences, labels, ids = list(zip(*items))
    batched_sequences = torch.tensor([0],dtype=torch.long)
    batched_labels = torch.tensor([0],dtype=torch.long)
    ##
    max_length = max([len(seq) for seq in sequences])
    padded_sequences = []
    for seq in sequences:
        nPads = max_length - len(seq)
        pads = torch.zeros(nPads, dtype=torch.long)
        padded_seq = torch.cat((pads, seq), dim=0)
        padded_sequences.append(padded_seq)
    batched_sequences = torch.stack(padded_sequences, dim=0)
    batched_labels = torch.stack(labels, dim=0)

    
    ##
    return batched_sequences, batched_labels, ids  

In [43]:
# TODO: à supprimer avant de soumettre
# Pour tester la fonction collate_fn
tempDataset = LSTMReviewDataset(tokenized_reviews, get_vocabulary_maxsize(tokenized_reviews, 100))
tempDataLoader = DataLoader(tempDataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
for batch_inputs, batch_labels, batch_ids in tempDataLoader:
    print(batch_inputs.shape, batch_labels.shape, len(batch_ids))
    break

torch.Size([32, 1326]) torch.Size([32, 2]) 32


12. **[2 points]** Écrivez une fonction *get_lstm_dataloader* qui prend en entrée *tokenized_reviews*, *vocab*, une taille de lot *batch_size*, un hyperparamètre *shuffle* et une fonction de *collate*. Elle doit utiliser la classe *LSTMReviewDataset* définie précédemment et retourner un objet de type *torch.utils.data.DataLoader* tenant compte des paramètres d’entrée.

In [47]:
def get_lstm_dataloader(tokenized_reviews:list[tuple[list[str], str]], vocab:dict[str,int], batch_size:int, shuffle:bool, collate_fn=collate_fn):
    ## TO DO
    data = LSTMReviewDataset(tokenized_reviews, vocab)
    dataloader = DataLoader(data, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)

    
    ##
    return dataloader

13. **[3 points]** Complétez la fonction *forward* de notre classificateur LSTM. Elle doit utiliser une boucle récurrente avec une cellule LSTM afin de produire des caractéristiques de classification basées sur le LSTM. N’utilisez que l’état caché final comme caractéristiques d’entrée pour la *final_classifier_layer*, qui sera un classificateur de régression logistique.

In [48]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, output_dim):
        super(LSTMClassifier, self).__init__()
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.embeddings = nn.Embedding(self.vocab_size, self.emb_dim, padding_idx=0)
        self.lstm = nn.LSTMCell(self.emb_dim, self.hidden_dim)
        self.final_classifier_layer = LogisticRegression(self.hidden_dim, self.output_dim)

    def forward(self, input): # |batch_size| x |sequence_length| x |emb_dim|
        batch_size = input.size()[0]
        sequence_length = input.size()[1]
        #our initial h_(i-1) and c_(i-1) vectors are dummies
        h_prior = torch.zeros(batch_size, self.hidden_dim)
        c_prior = torch.zeros(batch_size, self.hidden_dim)
        ## TO DO
        input_t = torch.transpose(input, 0, 1) # |sequence_length| x |batch_size| x |emb_dim|
        for i in range(sequence_length):
            x_i = self.embeddings(input_t[i])
            h_prior, c_prior = self.lstm(x_i, (h_prior, c_prior))
        

         # Use the final hidden state for classification
        output = self.final_classifier_layer(h_prior)
        
        ##
        return output 

#### Entraînons notre classificateur basé sur un LSTM !

L’entraînement du modèle prendra un certain temps — prévoyez environ 15 minutes sur un ordinateur portable de gamme moyenne.

In [49]:
def train_LR_lstm_classifier(train_dataset:list[tuple[str, str]], review_classes:list[str], batch_size = 32): 
    tokenized_train = [(preprocess(review), review_class) for review, review_class in  train_dataset]
    vocab = get_vocabulary_maxsize(tokenized_train, 10000)
    
    train_dataloader = get_lstm_dataloader(tokenized_train, vocab, batch_size, shuffle=True)

    model = LSTMClassifier(len(vocab), 64, 64, len(review_classes))
    trained_model = train(model, train_dataloader)
    return vocab, trained_model

In [50]:
lstm_vocab, lstm_model = train_LR_lstm_classifier(train_dataset, review_classes)

Epoch 1:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/50 [00:00<?, ?it/s]

Nous pouvons évaluer notre classificateur basé sur un LSTM à l’aide de la même fonction *eval()* définie précédemment. Nous l’évaluerons à la fois sur les données de test et d’entraînement afin de comparer les performances du modèle.

In [29]:
def eval_LR_lstm_classifier(test_dataset:list[tuple[str, str]], lstm_vocab:list[tuple], model, collate_fn=collate_fn):
    tokenized_test = [(preprocess(review), review_class) for review, review_class in  test_dataset]
    test_dataloader = get_lstm_dataloader(tokenized_test, lstm_vocab, batch_size=32, shuffle=False, collate_fn=collate_fn)
    results = eval(model, test_dataloader)
    return results

In [30]:
lstm_test_results = eval_LR_lstm_classifier(test_dataset, lstm_vocab, lstm_model)
lstm_train_results = eval_LR_lstm_classifier(train_dataset, lstm_vocab, lstm_model)
lstm_accuracies = {'lstm test':get_accuracy_scores(lstm_test_results),
                  'lstm train':get_accuracy_scores(lstm_train_results)}
pd.DataFrame.from_dict(lstm_accuracies, orient='index')

,precision,recall,f1,accuracy
lstm test,0.623529,0.53000,0.572973,0.60500
lstm train,0.911517,0.81125,0.858466,0.86625


In [ ]:
#Que remarquez-vous concernant la relation entre la précision, le rappel et le score F1 dans cette tâche ? Pourquoi cela pourrait-il être le cas ? Écrivez 1 ou 2 phrases expliquant votre observation.
# Les métriques précision, rappel et F1 sont toutes relativement élevées et proches les unes des autres, ce qui suggère que le modèle a un bon équilibre entre la précision et le rappel. Cela pourrait être dû au fait que les critiques de films ont des caractéristiques distinctes qui permettent au modèle de les différencier efficacement, ou que le modèle a été bien entraîné pour capturer les nuances du langage utilisé dans les critiques.

## Questions bonus

14. **[1 point bonus]** Que remarquez-vous concernant la relation entre la précision, le rappel et le score F1 dans cette tâche ? Pourquoi cela pourrait-il être le cas ? Écrivez 1 ou 2 phrases expliquant votre observation.

In [ ]:
#Les métriques d'entraînement sont meilleures que les métriques de test, ce qui suggère que le modèle a peut-être surappris les données d'entraînement. Cependant, les métriques de test ne sont pas catastrophiques, ce qui indique que le modèle a tout de même appris des représentations utiles pour la classification.
#Toutes les métriques ont des valeurs de précision, rappel, F1 et accuracy similaires, ce qui suggère que le modèle a un bon équilibre entre la précision et le rappel, et qu'il est globalement performant pour la classification des critiques de films.

Les métriques ont des valeurs similaires.

15. **[1 point bonus]** Qu’observez-vous à propos des résultats du LSTM ? Selon vous, à quel point ce modèle serait-il performant pour classifier de nouvelles critiques du monde réel comparativement, par exemple, au classificateur bigramme ? Écrivez 2 à 3 phrases expliquant vos observations et votre hypothèse concernant la performance du modèle sur des données nouvelles et inédites.